[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/09_naive_bayes/01_bayes_and_naive_assumption.ipynb)

# Bài 01 — Định lý Bayes và giả định "ngây thơ"

**🎯 Mục tiêu**
- Ôn định lý Bayes qua ví dụ xét nghiệm y tế — vì sao trực giác thường sai (base rate fallacy).
- Hiểu **model sinh (generative)** vs **model phân biệt (discriminative)**.
- Phát biểu giả định **độc lập có điều kiện** của Naive Bayes và hệ quả: số tham số giảm từ hàm mũ xuống tuyến tính.
- Tính tay một bộ phân loại Naive Bayes; vì sao phải tính trong **không gian log**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/09_naive_bayes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()

## 1. Định lý Bayes

$$P(y \mid \mathbf x) = \frac{P(\mathbf x \mid y)\,P(y)}{P(\mathbf x)} \qquad\text{posterior} = \frac{\text{likelihood} \times \text{prior}}{\text{evidence}}$$

**Ví dụ kinh điển.** Một bệnh có tỉ lệ mắc 1%. Xét nghiệm có độ nhạy 95% ($P(+ \mid \text{bệnh}) = 0.95$) và tỉ lệ dương tính giả 5% ($P(+ \mid \text{khoẻ}) = 0.05$). Một người xét nghiệm dương tính — xác suất người đó thật sự mắc bệnh là bao nhiêu? Đa số trả lời "95%"...

In [ ]:
prior = 0.01
sensitivity, false_positive = 0.95, 0.05
evidence = sensitivity * prior + false_positive * (1 - prior)          # P(+)
posterior = sensitivity * prior / evidence
print(f"P(bệnh | +) = {posterior:.3f}")

# kiểm chứng bằng mô phỏng 1 triệu người
rng = np.random.default_rng(0)
sick = rng.random(1_000_000) < prior
positive = np.where(sick, rng.random(sick.size) < sensitivity, rng.random(sick.size) < false_positive)
print(f"mô phỏng: {sick[positive].mean():.3f}")

Chỉ ~16%! Vì người khoẻ đông gấp 99 lần, 5% dương tính giả của họ (≈ 4 950 người / 100 000) áp đảo 95% dương tính thật của người bệnh (≈ 950 người). **Prior quan trọng** — bỏ qua nó là *base rate fallacy*. Xét nghiệm lần hai dương tính tiếp? Posterior lần một trở thành prior lần hai:

In [ ]:
p = prior
for test in range(1, 4):
    p = sensitivity * p / (sensitivity * p + false_positive * (1 - p))
    print(f"sau {test} lần dương tính: P(bệnh) = {p:.3f}")

Ý nghĩa của **cùng một** kết quả dương tính phụ thuộc mạnh vào prior — tỉ lệ mắc bệnh của nhóm được xét nghiệm:

In [ ]:
priors = np.logspace(-4, np.log10(0.5), 200)
posteriors = sensitivity * priors / (sensitivity * priors + false_positive * (1 - priors))
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(priors, posteriors, lw=2)
ax.scatter([prior], [posterior], color="C3", zorder=3, label=f"tỉ lệ mắc 1% → P(bệnh | +) = {posterior:.2f}")
ax.axhline(0.5, color="0.6", ls=":")
ax.set_xscale("log")
ax.set_xlabel("prior: tỉ lệ mắc bệnh (thang log)")
ax.set_ylabel("P(bệnh | dương tính)")
ax.set_title("Cùng xét nghiệm (độ nhạy 95%, dương tính giả 5%)")
ax.legend()
plt.show()
print(f"P(bệnh | +) vượt 50% khi tỉ lệ mắc từ khoảng {priors[np.argmax(posteriors >= 0.5)]:.1%} trở lên")

Sàng lọc diện rộng (prior thấp) sinh rất nhiều dương tính giả; cùng xét nghiệm đó dùng cho nhóm đã có triệu chứng (prior cao) thì đáng tin hơn hẳn. Trong ML cũng vậy: precision của một bộ phân loại phụ thuộc **tỉ lệ lớp dương** của dữ liệu triển khai (chương 14, 19).

## 2. Model sinh vs model phân biệt

| | Generative (sinh) | Discriminative (phân biệt) |
|---|---|---|
| Mô hình hoá | $P(\mathbf x \mid y)$ và $P(y)$ → suy ra $P(y \mid \mathbf x)$ qua Bayes | trực tiếp $P(y \mid \mathbf x)$ hoặc ranh giới |
| Ví dụ | Naive Bayes, LDA/QDA, GMM, HMM | Logistic Regression, SVM, cây, mạng neural |
| Ưu | học nhanh với ít dữ liệu, sinh được dữ liệu mới, xử lý missing tự nhiên | thường chính xác hơn khi nhiều dữ liệu |
| Nhược | giả định về phân phối của $\mathbf x$ có thể sai | cần nhiều dữ liệu hơn, không mô hình hoá $\mathbf x$ |

Phân loại theo **MAP** (maximum a posteriori): $P(\mathbf x)$ không phụ thuộc $y$ nên

$$\hat y = \arg\max_c P(y = c \mid \mathbf x) = \arg\max_c P(\mathbf x \mid y = c)\,P(y = c)$$

## 3. Giả định "ngây thơ"

Vấn đề: ước lượng $P(\mathbf x \mid y)$ với $d$ feature nhị phân cần $2^d - 1$ tham số **cho mỗi lớp** — với $d = 30$ là hơn 1 tỉ, không bao giờ đủ dữ liệu.

**Naive Bayes giả định:** các feature **độc lập có điều kiện** khi biết lớp:

$$P(\mathbf x \mid y) = \prod_{j=1}^{d} P(x_j \mid y) \quad\Rightarrow\quad \hat y = \arg\max_c \left[\log P(y = c) + \sum_{j=1}^d \log P(x_j \mid y = c)\right]$$

Số tham số giảm còn $d$ mỗi lớp. Giả định này **gần như luôn sai** (từ "bóng" và "đá" trong email không độc lập) — nhưng model vẫn thường phân loại tốt (bài 04 giải thích vì sao).

In [ ]:
for d in [5, 10, 20, 30, 50]:
    print(f"d = {d:2d}: đầy đủ {2**d - 1:>20,} tham số/lớp | naive {d:3d} tham số/lớp")

## 4. Tính tay: lọc thư rác

Tập train 8 email, 3 feature nhị phân (email có chứa từ đó hay không):

In [ ]:
emails = pd.DataFrame({
    "miễn_phí":   [1, 1, 1, 0, 0, 0, 1, 0],
    "khuyến_mãi": [1, 0, 1, 1, 0, 0, 0, 0],
    "họp":        [0, 0, 0, 0, 1, 1, 1, 1],
    "spam":       [1, 1, 1, 1, 0, 0, 0, 0],
})
emails

Email mới: có "miễn_phí", không có "khuyến_mãi", có "họp" → $\mathbf x = (1, 0, 1)$. Tính:

$$P(\text{spam}) \cdot P(x_1 = 1 \mid \text{spam}) \cdot P(x_2 = 0 \mid \text{spam}) \cdot P(x_3 = 1 \mid \text{spam})$$

và tương tự cho "không spam".

In [ ]:
x_new = {"miễn_phí": 1, "khuyến_mãi": 0, "họp": 1}
scores = {}
for c in [1, 0]:
    sub = emails[emails.spam == c]
    score = len(sub) / len(emails)                                  # prior
    terms = [f"{score:.2f}"]
    for word, value in x_new.items():
        p_word = (sub[word] == value).mean()                        # P(x_j = value | y = c)
        score *= p_word
        terms.append(f"{p_word:.2f}")
    scores[c] = score
    print(f"y = {c}: " + " × ".join(terms) + f" = {score:.4f}")
total = sum(scores.values())
print(f"P(spam | x) = {scores[1] / total:.3f}")

Phép tính trên có lỗ hổng: nếu một từ **không bao giờ** xuất hiện trong spam ở tập train (vd "họp" — $P(\text{họp} = 1 \mid \text{spam}) = 0$), cả tích bằng 0 bất kể các bằng chứng khác mạnh thế nào → **zero-frequency problem**. Cách chữa: **Laplace smoothing** — cộng thêm $\alpha$ (thường 1) vào mỗi bộ đếm:

$$P(x_j = v \mid y = c) = \frac{N_{c,j,v} + \alpha}{N_c + \alpha \cdot |V_j|}$$

với $|V_j|$ là số giá trị có thể của feature $j$ (ở đây 2).

In [ ]:
alpha = 1.0
scores_smooth = {}
for c in [1, 0]:
    sub = emails[emails.spam == c]
    score = len(sub) / len(emails)
    for word, value in x_new.items():
        score *= ((sub[word] == value).sum() + alpha) / (len(sub) + 2 * alpha)
    scores_smooth[c] = score
print(f"không smoothing: P(spam | x) = {scores[1] / sum(scores.values()):.3f}")
print(f"Laplace (α = 1): P(spam | x) = {scores_smooth[1] / sum(scores_smooth.values()):.3f}")

## 5. Vì sao phải tính trong không gian log?

Tích của hàng nghìn xác suất nhỏ (văn bản có hàng nghìn từ) **tràn dưới** (*underflow*) về 0 trong số thực 64-bit:

In [ ]:
probs = np.full(1000, 0.01)                 # 1000 từ, mỗi từ xác suất 0.01
print("tích trực tiếp:", np.prod(probs))    # 1e-2000 → 0.0
print("tổng log      :", np.log(probs).sum())

Tính $\log P(y) + \sum_j \log P(x_j \mid y)$ thay cho tích. Để đổi về xác suất posterior, dùng **log-sum-exp** (trừ max trước khi `exp`, giống softmax ổn định ở chương 07):

$$P(y = c \mid \mathbf x) = \frac{e^{s_c}}{\sum_k e^{s_k}} = \exp\!\Big(s_c - \operatorname{logsumexp}(\mathbf s)\Big)$$

In [ ]:
from scipy.special import logsumexp

s = np.array([-2300.0, -2302.5])            # log-score của 2 lớp
with np.errstate(invalid="ignore"):
    print("exp trực tiếp:", np.exp(s) / np.exp(s).sum())      # 0/0 → nan
print("log-sum-exp  :", np.exp(s - logsumexp(s)))

## 🧠 Kiểm tra nhanh

1. Xét nghiệm có độ nhạy 99%, dương tính giả 1%, bệnh có tỉ lệ 0.1%. $P(\text{bệnh} \mid +)$ khoảng bao nhiêu?
2. Naive Bayes là model sinh hay phân biệt? Cái "naive" nằm ở đâu?
3. Vì sao không cần tính $P(\mathbf x)$ khi phân loại?
4. Không có Laplace smoothing, điều gì xảy ra khi email mới chứa một từ chưa từng xuất hiện trong spam?

## 📌 Tóm tắt

- Bayes: posterior ∝ likelihood × prior; prior (base rate) quan trọng.
- Naive Bayes = model sinh + giả định độc lập có điều kiện → $d$ tham số mỗi lớp thay vì $2^d$.
- Thực hành: Laplace smoothing chống zero-frequency; tính trong không gian log + log-sum-exp chống underflow.

## 📚 Tài liệu tham khảo

- Vũ Hữu Tiệp — [Naive Bayes Classifier](https://machinelearningcoban.com/2017/08/08/nbc/)
- Murphy — *Probabilistic Machine Learning: An Introduction*, §9.3
- Jurafsky & Martin — *Speech and Language Processing*, chương 4 (Naive Bayes và phân loại văn bản)